# Phase 4 - calibration, abstention, robustness

Everything below is read from `results/phase4.json`, which `src/phase4.py` writes by
running the four Phase 4 steps at **one** operating point fitted on validation only.

Two rules this notebook obeys:

* no number is recomputed here - if a table is wrong, the fix belongs in `src/`;
* every robustness row is a **probe, not a real-world claim**.


In [ ]:
import json
from pathlib import Path

import pandas as pd

# Works whether the notebook is run from the repo root or from notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
payload = json.loads((ROOT / "results/phase4.json").read_text(encoding="utf-8"))
h = payload["headline"]
print("model:", h["model"])
print(f"temperature = {h['temperature']:.4f}  (raw-val fit {h['temperature_raw_val']:.4f})")
print(f"t_high = {h['t_high']:.4f}   t_low = {h['t_low']:.4f}   band rows = {h['n_rows_in_band']}")
print(f"real safe val rows = {h['n_real_safe_val']}  |  synthetic safe val rows = {h['n_synth_safe_val']}")


## 1. Novelty buckets - the honest generalisation view

Recall per `max_train_sim` bucket. `n` is printed on every row; buckets under 30 rows
are not claims.


In [ ]:
rows = []
for target, buckets in payload["novelty"]["targets"].items():
    for bucket, m in buckets.items():
        rows.append({"stratum": target, "max_train_sim": bucket, **m})
novelty = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(novelty[["stratum", "max_train_sim", "n", "n_scam", "n_safe",
               "scam_recall", "fpr", "auroc", "too_small_for_claims"]].to_string(index=False))


## 2. Calibration - ECE before and after temperature scaling

One temperature, fitted on reweighted validation (equal weight per scam/safe x
real/synthetic cell). Test strata are only ever scored with it.


In [ ]:
cal = payload["calibration"]
targets = pd.DataFrame(cal["targets"])
print("validation cells:", cal["cell_counts"])
print(f"ECE (val, raw)        = {cal['ece_val_raw']:.4f}")
print(f"ECE (val, calibrated) = {cal['ece_val_calibrated']:.4f}")
print()
print(targets[["target", "n_rows", "n_safe", "ece_raw", "ece_calibrated",
               "mean_p_raw", "mean_p_calibrated"]].to_string(index=False))


## 3. The abstention band

`scam` if p >= t_high, `safe` if p <= t_low, otherwise `suspicious`.


In [ ]:
th = cal["thresholds"]
print("band available:", th["abstention_available"], "| reason:", th["band_reason"])
print(f"precision at t_high = {th['precision_at_t_high']:.4f} "
      f"({th['n_scam_behind_t_high']} scam rows behind it)")
print(f"NPV at t_low        = {th['npv_at_t_low']:.4f} "
      f"({th['n_safe_behind_t_low']} safe rows behind it)")
print()
print(pd.DataFrame(cal["coverage_sweep"])[
    ["target", "t_high", "t_low", "band_rows", "band_share",
     "holdout_abstain_share", "holdout_scam_share"]].to_string(index=False))


In [ ]:
hold = cal["holdout"]
print(f"suspicious holdout: {hold['n_rows']} rows -> {hold['counts']}")
print(f"share abstained: {hold['share_suspicious']:.1%}   mean p: {hold['mean_p']:.3f}")


## 4. Error analysis - where the mistakes are

Counts per set; the 30 worst examples per direction live in `results/error_analysis.md`.


In [ ]:
errs = pd.DataFrame(payload["errors"])[
    ["target", "n_rows", "n_safe", "n_scam", "n_fp", "n_fn", "n_abstain"]]
print(errs.to_string(index=False))


## 5. Robustness probes - **probe, not a real-world claim**

### (a) Perturbation attacks at the fixed operating point


In [ ]:
atk = pd.DataFrame(payload["robustness"]["attacks"])
print(atk[["pool", "attack", "n", "n_changed", "recall", "recall_changed",
           "delta_recall", "mean_p"]].to_string(index=False))


### (b) Hinglish transfer - translation-derived, not real Hinglish

In [ ]:
t = payload["robustness"]["transfer"]
print(f"n={t['n']}  rows changed={t['share_changed']:.1%}  "
      f"tokens replaced={t['mean_token_replacement']:.1%}")
print(f"recall English        = {t['recall_english']:.3f}")
print(f"recall Hinglish-derived = {t['recall_hinglish']:.3f}")
print(t["gemini_status"])
for ex in t["examples"][:3]:
    print("\nEN:", ex["english"][:160])
    print("HI:", ex["hinglish"][:160])


### (c) Fresh safe chat - false positive rate

In [ ]:
s = payload["robustness"]["safe_chat"]
print(f"FPR = {s['fpr']:.1%} ({s['n_flagged']}/{s['n']})")
print(pd.DataFrame(s["by_style"]).T)
print(f"max similarity to any train row: {s['max_train_sim']:.2f}")
for text in s["examples_flagged"]:
    print("-", text[:140])


## What I notice

1. **The decision is saturated, so the classic robustness story does not apply.**
   Recall sits at 0.99 on real scams and never drops more than 0.010 under any of the
   eight attacks - not because the model is robust, but because it calls almost
   everything a scam. The attack table measures a ceiling, not resilience.

2. **The false-positive side is where the model actually breaks.**
   15.3% of real English ham (n=59) and 13.0% of 200 fresh legitimate messages are
   flagged, and the English half of that probe is far worse (20%) than the Hinglish
   half (6%). A user-facing system with a 1-in-5 flag rate on ordinary English chat is
   unusable regardless of recall.

3. **The three-way verdict collapses to two-way on this model.**
   `t_high` and `t_low` land on the same cut (0.346) with zero validation rows between
   them, and no target from 0.80 to 0.99 opens a band. The 783-row human-`suspicious`
   holdout gets 0% abstention: 780 rows are pushed to a confident `scam`.

4. **Calibration is good where the data is, and bad where it matters.**
   Validation ECE falls 0.0269 -> 0.0201, but `public_ham_english` only improves from
   0.245 to 0.139, and the two n=5 user strata stay badly wrong (0.542). Temperature
   scaling cannot fix a stratum that is 59 rows of one corpus.

5. **Translation-derived Hinglish barely moves the score (0.990 -> 0.987).**
   That is *not* evidence the model handles real code-mixed traffic: the rewrite only
   replaces ~39% of tokens with dictionary forms, and no Gemini key was present, so the
   paraphrase path never ran. Real Hinglish remains untested.
